# CS02 — Machine Learning
## Análise de Risco de Equipamentos

##### Integrantes:

##### Adalberto Alves Cruz             | RM: 574115
##### Bruno Henrique Ferreira Ambrosio | RM: 571218
##### Gustavo da Silva Nascimento      | RM: 570821
##### Lucas Maximo dos Santos          | RM: 569714
##### Renan de Assis Rodrigues         | RM: 574049
##### Tiago Thomaz Cesaro              | RM: 569374

Nesse notebook analisamos dados de sensores de temperatura e umidade de equipamentos eletrônicos para identificar quais estão em situação de risco.

Etapas que segui:
- importar as bibliotecas e criar o dataset
- limpar e preparar os dados
- fazer a análise exploratória (EDA)
- criar a variável de risco
- treinar um modelo de classificação (árvore de decisão)

In [ ]:
# importando as bibliotecas necessarias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

equipamentos = ['EQ-001', 'EQ-002', 'EQ-003', 'EQ-004', 'EQ-005']
sensores = ['S1', 'S2', 'S3', 'S4', 'S5']

# configuracao de media e desvio de temp e umidade de cada equipamento
config = {
    'EQ-001': {'temp_mean': 38, 'temp_std': 4, 'umid_mean': 58, 'umid_std': 6},
    'EQ-002': {'temp_mean': 26, 'temp_std': 3, 'umid_mean': 55, 'umid_std': 5},
    'EQ-003': {'temp_mean': 28, 'temp_std': 3, 'umid_mean': 73, 'umid_std': 7},
    'EQ-004': {'temp_mean': 24, 'temp_std': 2, 'umid_mean': 50, 'umid_std': 5},
    'EQ-005': {'temp_mean': 31, 'temp_std': 3, 'umid_mean': 60, 'umid_std': 6},
}

data_inicio = datetime(2024, 1, 1)
registros = []

for i in range(600):
    # 2024 tem 366 dias (ano bissexto) = 8784 horas
    data = data_inicio + timedelta(hours=int(np.random.uniform(0, 8784)))
    equip = np.random.choice(equipamentos)
    sensor = np.random.choice(sensores)
    c = config[equip]
    temp = round(float(np.clip(np.random.normal(c['temp_mean'], c['temp_std']), 0, 60)), 1)
    umid = round(float(np.clip(np.random.normal(c['umid_mean'], c['umid_std']), 20, 100)), 1)
    registros.append({
        'data_medicao': data.strftime('%Y-%m-%d %H:%M:%S'),
        'equipamento': equip,
        'sensor': sensor,
        'temperatura': temp,
        'umidade': umid
    })

df = pd.DataFrame(registros)
# converter para datetime antes de ordenar — ordenar string de data pode dar erro em alguns formatos
df['data_medicao'] = pd.to_datetime(df['data_medicao'])
df = df.sort_values('data_medicao').reset_index(drop=True)

print('dataset criado com sucesso!')
print('linhas e colunas:', df.shape)
df.head(10)

## Dataset Simulado

Como não consegui fazer o upload do CSV funcionar no Colab, optei por simular os dados.

Os dados representam leituras de sensores em 5 equipamentos ao longo de 2024. O EQ-001 foi configurado com temperatura alta e o EQ-003 com umidade alta para simular situações reais de risco. Os outros três ficaram com valores normais para servir de comparação.

In [ ]:
# verificando os tipos de cada coluna
print(df.dtypes)

In [ ]:
# estatisticas descritivas das colunas numericas
df.describe()

## Limpeza dos dados

In [ ]:
# checando se tem valores nulos
print('Valores nulos por coluna:')
print(df.isnull().sum())

In [ ]:
# checando se tem linhas duplicadas
print('Linhas duplicadas:', df.duplicated().sum())

In [ ]:
# a conversao para datetime ja foi feita na criacao do dataset
# aqui so verifico se o tipo esta correto
print('Tipo da coluna data_medicao:', df['data_medicao'].dtype)

In [ ]:
# extraindo mes e hora para usar na analise
df['mes'] = df['data_medicao'].dt.month
df['hora'] = df['data_medicao'].dt.hour

print('Colunas novas criadas: mes e hora')
print('Colunas atuais:', df.columns.tolist())
df.head()

In [ ]:
def detectar_outliers_iqr(serie, nome):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    outliers = serie[(serie < lim_inf) | (serie > lim_sup)]
    print(f'{nome}: Q1={Q1:.1f} Q3={Q3:.1f} IQR={IQR:.1f} | Limites: [{lim_inf:.1f}, {lim_sup:.1f}]')
    print(f'  Outliers detectados: {len(outliers)} ({len(outliers)/len(serie)*100:.1f}%)\n')
    return lim_inf, lim_sup

print('detectando outliers com metodo IQR:\n')
lim_t_inf, lim_t_sup = detectar_outliers_iqr(df['temperatura'], 'Temperatura (C)')
lim_u_inf, lim_u_sup = detectar_outliers_iqr(df['umidade'], 'Umidade (%)')

# decidi manter os outliers porque eles representam as situacoes de risco
# que queremos que o modelo aprenda a identificar
df['outlier_temp'] = ((df['temperatura'] < lim_t_inf) | (df['temperatura'] > lim_t_sup)).astype(int)
df['outlier_umid'] = ((df['umidade'] < lim_u_inf) | (df['umidade'] > lim_u_sup)).astype(int)

print(f'medicoes com outlier de temperatura: {df["outlier_temp"].sum()}')
print(f'medicoes com outlier de umidade: {df["outlier_umid"].sum()}')

In [ ]:
# valores unicos nas colunas categoricas
print('Equipamentos unicos:', df['equipamento'].unique())
print('Sensores unicos:', df['sensor'].unique())
print()
print('Quantidade de medicoes por equipamento:')
print(df['equipamento'].value_counts())

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_equip = LabelEncoder()
le_sensor = LabelEncoder()

df['equipamento_num'] = le_equip.fit_transform(df['equipamento'])
df['sensor_num'] = le_sensor.fit_transform(df['sensor'])

print('variaveis categoricas transformadas com sucesso!')
df[['equipamento', 'equipamento_num', 'sensor', 'sensor_num']].head(8)

## Conversão das variáveis categóricas para número

Usei o `LabelEncoder` do sklearn para transformar os nomes dos equipamentos e sensores em números, já que o modelo não trabalha com texto.

Essa é a forma mais correta de fazer esse tipo de conversão em projetos de Machine Learning.

## Análise Exploratória dos Dados

In [ ]:
# histograma da temperatura
plt.figure(figsize=(8, 4))
plt.hist(df['temperatura'], bins=15, color='tomato', edgecolor='black')
plt.title('Distribuicao da Temperatura')
plt.xlabel('Temperatura (graus C)')
plt.ylabel('Numero de medicoes')
plt.tight_layout()
plt.show()

In [ ]:
# histograma da umidade
plt.figure(figsize=(8, 4))
plt.hist(df['umidade'], bins=15, color='steelblue', edgecolor='black')
plt.title('Distribuicao da Umidade')
plt.xlabel('Umidade (%)')
plt.ylabel('Numero de medicoes')
plt.tight_layout()
plt.show()

In [ ]:
# boxplot temperatura por equipamento
plt.figure(figsize=(9, 5))
sns.boxplot(x='equipamento', y='temperatura', hue='equipamento', data=df, palette='Reds', legend=False)
plt.title('Temperatura por Equipamento')
plt.xlabel('Equipamento')
plt.ylabel('Temperatura (graus C)')
plt.axhline(35, color='darkred', linestyle='--', label='Limite critico (35 graus C)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# boxplot umidade por equipamento
plt.figure(figsize=(9, 5))
sns.boxplot(x='equipamento', y='umidade', hue='equipamento', data=df, palette='Blues', legend=False)
plt.title('Umidade por Equipamento')
plt.xlabel('Equipamento')
plt.ylabel('Umidade (%)')
plt.axhline(70, color='navy', linestyle='--', label='Limite critico (70%)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# dispersao temperatura vs umidade separado por equipamento
plt.figure(figsize=(9, 5))
for equip in df['equipamento'].unique():
    grupo = df[df['equipamento'] == equip]
    plt.scatter(grupo['temperatura'], grupo['umidade'], label=equip, s=60)
plt.title('Temperatura vs Umidade por Equipamento')
plt.xlabel('Temperatura (graus C)')
plt.ylabel('Umidade (%)')
plt.legend()
plt.show()

In [ ]:
# numero de medicoes por Equipamento
plt.figure(figsize=(7, 4))
df['equipamento'].value_counts().plot(kind='bar', color='orange', edgecolor='black')
plt.title('Numero de Medicoes por Equipamento')
plt.xlabel('Equipamento')
plt.ylabel('Quantidade')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# medicoes ao longo dos meses
medicoes_mes = df.groupby('mes').size()
plt.figure(figsize=(7, 4))
medicoes_mes.plot(kind='bar', color='mediumpurple', edgecolor='black')
plt.title('Medicoes por Mes')
plt.xlabel('Mes')
plt.ylabel('Quantidade')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# mapa de correlacao entre variaveis numericas
colunas = ['temperatura', 'umidade', 'equipamento_num', 'sensor_num', 'mes', 'hora', 'outlier_temp', 'outlier_umid']
corr = df[colunas].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Mapa de Correlacao entre Variaveis')
plt.tight_layout()
plt.show()

## Variável de Risco

Aqui criei uma coluna chamada `risco` para usar como alvo no modelo.

Critério:
- `risco = 1` se temperatura > 35 graus C **OU** umidade > 70%
- `risco = 0` caso contrário

Esses limites são baseados nas condições ideais de operação de equipamentos eletrônicos.

In [ ]:
# criando a variavel risco
df['risco'] = 0
df.loc[(df['temperatura'] > 35) | (df['umidade'] > 70), 'risco'] = 1

print('Distribuicao da variavel risco:')
print(df['risco'].value_counts())
print()
porcentagem = round(df['risco'].mean() * 100, 1)
print(str(porcentagem) + '% das medicoes estao em situacao de risco')

In [ ]:
# percentual de risco por equipamento
risco_equip = df.groupby('equipamento')['risco'].mean() * 100
risco_equip = risco_equip.round(1).sort_values(ascending=False)

cores = ['red' if v >= 60 else 'orange' if v >= 30 else 'green' for v in risco_equip.values]

plt.figure(figsize=(8, 4))
risco_equip.plot(kind='bar', color=cores, edgecolor='black')
plt.title('Percentual de Medicoes em Risco por Equipamento')
plt.xlabel('Equipamento')
plt.ylabel('% em risco')
plt.xticks(rotation=0)
plt.show()

print(risco_equip)

In [ ]:
# medias por equipamento (resumo final)
medias = df.groupby('equipamento')[['temperatura', 'umidade', 'risco']].mean().round(2)
print('Medias por equipamento:')
print(medias)

## Insights da Análise

Olhando os gráficos, consegui identificar o seguinte:

- O **EQ-001** é o equipamento com maior risco de temperatura — frequentemente passa de 35 graus
- O **EQ-003** tem umidade alta — os valores ficam bastante acima de 70% em várias medições
- Os equipamentos **EQ-002, EQ-004 e EQ-005** estão em condições normais e servem de comparação
- Temperatura e umidade parecem ser independentes entre si, então os dois fatores precisam ser analisados juntos

As variáveis que mais parecem influenciar o risco são a temperatura e qual equipamento está sendo medido.

## Modelo de Classificação — Árvore de Decisão

Aqui apliquei uma árvore de decisão para classificar as medições em risco ou sem risco.

O dataset foi dividido em 80% para treino e 20% para teste.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score,
                             precision_score, recall_score, f1_score)

# o objetivo aqui nao e prever o risco a partir das leituras de sensor,
# mas verificar se informacoes operacionais (equipamento, sensor, mes e hora)
# conseguem indicar padroes associados ao risco
features = ['equipamento_num', 'sensor_num', 'mes', 'hora']
X = df[features]
y = df['risco']

# divisao treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('tamanho treino:', X_train.shape)
print('tamanho teste: ', X_test.shape)
print()
print('distribuicao de risco no treino:')
print(y_train.value_counts())

# treinando o modelo
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

# avaliando no teste
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print()
print('acuracia:', round(accuracy_score(y_test, y_pred) * 100, 1), '%')
print('roc-auc: ', round(roc_auc_score(y_test, y_prob), 3))
print()
print('Precisao:', round(precision_score(y_test, y_pred), 3))
print('Recall:  ', round(recall_score(y_test, y_pred), 3))
print('F1-Score:', round(f1_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred, target_names=['Sem Risco', 'Com Risco']))

# validacao cruzada para checar se o resultado e consistente
cv = cross_val_score(clf, X, y, cv=5)
print(f'validacao cruzada (5-fold): {cv.mean()*100:.1f}% +/- {cv.std()*100:.1f}%')

# matriz de confusao
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Sem Risco', 'Com Risco'],
            yticklabels=['Sem Risco', 'Com Risco'])
plt.title('Matriz de Confusao - Arvore de Decisao')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

# visualizacao da arvore de decisao
plt.figure(figsize=(18, 8))
plot_tree(clf, feature_names=features, class_names=['Sem Risco', 'Com Risco'],
          filled=True, rounded=True, max_depth=3)
plt.title('Arvore de Decisao (ate profundidade 3)')
plt.tight_layout()
plt.show()

# importancia das variaveis
importancias = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=True)
plt.figure(figsize=(7, 3))
importancias.plot(kind='barh', color='teal', edgecolor='black')
plt.title('Importancia das Variaveis')
plt.xlabel('Importancia (Gini)')
plt.tight_layout()
plt.show()

print('ranking de importancia:')
print(importancias.sort_values(ascending=False).to_string())

mais_importante = importancias.idxmax()
print()
print('CONCLUSAO:')
print(f'A variavel mais importante para a arvore foi {mais_importante}.')
print('Isso sugere que determinados equipamentos ou periodos apresentam maior propensao a ocorrencias de risco.')